# Team Epic 
# Telecommunication - Customer Churn Analysis

## Table of Content

* [Importing Liberaries](#h0) 
* [Description of Dataset](#h1)
* [Handling NaN values](#h2)
* [Data Pre-Processing](#h3)
* [Modeling](#h4)
* [Predictions and Evaluations](#h5) 
* [Final Model](#h6)
* [Final Prediction](#h7)



### Importing tha required liberaries <a class= "anchor" id="h0"></a>

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
import pickle

### Loading the Data set

In [ ]:
churn_df = pd.read_csv("Customer-Churn-Prediction.csv")
churn_df

### Feature description of dataset<a class= "anchor" id="h1"></a>

In [ ]:
churn_df.info()

### Handling missing values <a class= "anchor" id="h2"></a>

In [ ]:
cat_col = ['international_plan', 'voice_mail_plan', 'area_code', 'state']
for i in cat_col:
    churn_df[i]= churn_df[i].astype('category').cat.codes

In [ ]:
churn_df.isnull().sum()

### Replacing NULL values in numerical Columns using Median

In [ ]:
for col in churn_df.columns[~churn_df.columns.isin(['state','area_code','churn'])]:
    churn_df[col] =  churn_df[col].fillna(churn_df[col].median())

### Replacing NULL values in categorical columns with mode.

In [ ]:
for col in churn_df[['state','area_code','churn']]:
    churn_df[col] =  churn_df[col].fillna(churn_df[col].mode()[0])

#### Checking for the more nan values

In [ ]:
churn_df.isnull().sum()

### Imbalancing of data 

In [ ]:
(churn_df['churn'].value_counts().index[0], churn_df['churn'].value_counts().values[0])

In [ ]:
(churn_df['churn'].value_counts().index[1], churn_df['churn'].value_counts().values[1])

### Converitng traget variable into numerical by Label Encoder

In [ ]:
X = []
X = churn_df.drop(['churn'],axis=1)
#features = pd.DataFrame(df_churn_pd).iloc[:,:-1]

label_churn = pd.DataFrame(churn_df, columns = ['churn'])
le = LabelEncoder()
label = churn_df['churn']

label = le.fit_transform(label)
print("Encoded values of churn after LabelEncoding: "  + str(label))

### Splitting tha balanced dataset using train_test_split.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, label, random_state= 0, test_size= 0.3)

In [ ]:
print("Size of the traintest :", X_train.shape, y_train.shape)
print("Size of the testset :", X_test.shape , y_test.shape)

## Pre-Processing<a class= "anchor" id="h3"></a>

In [ ]:
cat_col = ['area_code','international_plan','voice_mail_plan']
num_col = X.select_dtypes(include=["float64"]).columns

pre_process = ColumnTransformer(transformers=[('cat', SimpleImputer(strategy='most_frequent'), cat_col),
                            ('num', SimpleImputer(strategy='median'), num_col)], 
                            remainder = "passthrough")

# Feature Scaling
scale_transformer = ColumnTransformer([('scale', StandardScaler(), slice(5,18))])

# Principal Component Analysis
pca_col = ColumnTransformer([('PCA', PCA(n_components=10), slice(0,18))])

# Classifier
lr_model = LogisticRegression()                                         
rf_model = RandomForestClassifier()
knn_model = KNeighborsClassifier(n_neighbors = 5)

### Modeling<a class= "anchor" id="h4"></a>

In [ ]:
pipe_model_lr = Pipeline([('pre_process', pre_process),
                           ('scale_transformer', scale_transformer),
                           ('pca_col', pca_col),
                           ('classifier1', lr_model)])

pipe_model_rf = Pipeline([('pre_process', pre_process),
                          ('scale_transformer', scale_transformer),
                          ('pca_col', pca_col),
                          ('classifier2', rf_model)])

pipe_model_knn = Pipeline([('pre_process', pre_process),
                 ('scale_transformer', scale_transformer),
                 ('pca_col', pca_col),
                 ('classifier3', knn_model)])


In [ ]:
pipeline = [pipe_model_lr, pipe_model_rf, pipe_model_knn]

In [ ]:
pipe_dict={0:'Logistic Regression',1:'Random Forest',2:'K Neighbors'}

# Fitting pipelines
for pipe in pipeline:
    pipe.fit(X_train,y_train)

In [ ]:
for i, model in enumerate(pipeline):
    print("{} Accuracy : {}".format(pipe_dict[i],model.score(X_test,y_test)*100))
    print("______________________________________________________")

### Predictions and Evaluations<a class= "anchor" id="h5"></a>

In [ ]:
for i,model in enumerate(pipeline):
    pred=model.predict(X_test)
    print("\n{} Classification Report :\n {}".format(pipe_dict[i],classification_report(y_test,pred)))
    print(" ______________________________________________________")

### Taking Best Model 

In [ ]:
best_accuracy =0
best_pipeline= 0
best_classifier=''
for i,model in enumerate(pipeline):
    if model.score(X_test,y_test)>best_accuracy:
        best_accuracy=model.score(X_test,y_test)
        best_pipeline=model
        best_classifier=i
print('Classifier with the best accuracy : {}'.format(pipe_dict[best_classifier]))

### Hyperparameters Tunning for best model using Random SearchCV using pipeline

In [ ]:
r_grid = [{'classifier2__n_estimators': range(100, 501, 100),
           "classifier2__max_depth": range(10,30,10),
          'classifier2__max_features': ['auto', 'log2'],
          "classifier2__min_samples_leaf": [1, 2, 4],
           "classifier2__min_samples_split": [2, 5, 10],
          }]

rfr_random = RandomizedSearchCV(estimator=pipe_model_rf, 
                                param_distributions= r_grid, 
                                cv = 10, verbose=2, 
                                 n_jobs=-1)

rfr_random.fit(X_train,y_train)
rfr_random.best_params_

### Final Model<a class= "anchor" id="h6"></a>

In [ ]:
rfr_random_model = Pipeline([('pre_process', pre_process),
                          ('scale_transformer', scale_transformer),
                          ('pca_col', pca_col),
                          ('classifier2', RandomForestClassifier(max_depth=20,
                                                                 min_samples_leaf=1,
                                                                 min_samples_split=2,
                                                                 n_estimators=500,
                                                                max_features = 'auto'))
                         ])

rfr_random_model.fit(X_train,y_train)

### Saving the model

In [ ]:
filename = 'model_v1.pkl'
pickle.dump(rfr_random_model, open(filename, 'wb'))

### Loading saved the model 

In [ ]:
filename = 'model_v1.pkl'
loaded_model = pickle.load(open(filename, 'rb'))
result = loaded_model.score(X_test, y_test)
print(result)

### Final Prediction<a class= "anchor" id="h7"></a>

In [ ]:
prediction = loaded_model.predict(X_test)
prediction

### Model Evaluation 

In [ ]:
cfm = confusion_matrix(y_test, prediction)
print("confusion matrix: \n")
print(cfm)
print('\n')
print("classification_report: \n", classification_report(y_test, prediction))